# GPR Buried-Object Detection: Phase 2 Benchmarking & Grouped 5-Fold Ablations (Google Colab)

### Instructions (2 minutes setup):
1. **Enable Free GPU**: In top menu, click **Runtime** → **Change runtime type** → select **T4 GPU** → **Save**.
2. **Upload Bundle**: Click the **Folder icon (📁)** on the left sidebar → Drag & drop `colab_bundle.zip` directly into the panel.
3. **Run**: Click **Runtime** → **Run all** (or **Ctrl + F9**).

When finished, **`results.zip` will automatically download to your computer!**

In [ ]:
# 1. GPU Environment Verification & Package Setup
!nvidia-smi
!pip install -q ultralytics

import torch
import ultralytics
print(f"PyTorch: {torch.__version__} | Ultralytics: {ultralytics.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    raise SystemError("GPU not detected! Please go to Runtime -> Change runtime type -> select T4 GPU.")

In [ ]:
# 2. Extract colab_bundle.zip (Takes ~3-5 seconds)
import os
import shutil
from pathlib import Path

bundle_path = Path("/content/colab_bundle.zip")
if not bundle_path.exists():
    raise FileNotFoundError("colab_bundle.zip not found in /content/! Please upload colab_bundle.zip via the left Files panel.")

repo_dir = Path("/content/repo")
if repo_dir.exists():
    shutil.rmtree(repo_dir)
repo_dir.mkdir(parents=True, exist_ok=True)

print("Extracting colab_bundle.zip into /content/repo...")
!unzip -q /content/colab_bundle.zip -d /content/repo

os.chdir(repo_dir)
print(f"Ready! Current directory: {os.getcwd()}")
!ls -la

In [ ]:
# 3. Run Phase 2 Model Benchmarking (YOLOv8n, YOLOv5nu, YOLO11n, YOLOv8s across 5 folds)
# Includes Gate 2 verification comparing Colab GPU YOLOv8n vs. Laptop CPU baseline
!python experiments/02_train_all_models.py --epochs 50 --device 0 --models yolov8n yolov5nu yolo11n yolov8s

In [ ]:
# 4. Run Grouped 5-Fold Leave-One-Out (LOO) & No-Augmentation Ablation Study
!python experiments/02_run_ablations.py --epochs 50 --device 0

In [ ]:
# 5. Display Model Comparison & Ablation Tables
import os
import pandas as pd
from pathlib import Path
from IPython.display import display

comp_csv = Path("/content/repo/results/model_comparison/comparison.csv")
if comp_csv.exists():
    print("\n" + "="*60)
    print("MODEL COMPARISON BENCHMARK (GROUPED 5-FOLD)")
    print("="*60)
    display(pd.read_csv(comp_csv))
else:
    print(f"{comp_csv} not found yet.")

abl_csv = Path("/content/repo/results/ablation_study/ablation_summary.csv")
if abl_csv.exists():
    print("\n" + "="*60)
    print("LEAVE-ONE-OUT ABLATION STUDY (GROUPED 5-FOLD)")
    print("="*60)
    display(pd.read_csv(abl_csv))
else:
    print(f"{abl_csv} not found yet.")

In [ ]:
# 6. Archive all results into /content/results.zip & Automatic Browser Download
import os
import zipfile
from pathlib import Path
from google.colab import files

results_dir = Path("/content/repo/results")
zip_dest = Path("/content/results.zip")

if not results_dir.exists():
    print(f"[WARNING] {results_dir} does not exist yet. Please check training progress.")
else:
    print(f"Archiving {results_dir} into {zip_dest}...")
    with zipfile.ZipFile(zip_dest, "w", zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files_list in os.walk(results_dir):
            for file in files_list:
                abs_path = Path(root) / file
                rel_path = abs_path.relative_to("/content/repo")
                zipf.write(abs_path, rel_path)

    size_mb = zip_dest.stat().st_size / (1024 * 1024)
    print(f"Done! Created {zip_dest} ({size_mb:.2f} MB).")
    print("Triggering browser download of results.zip...")
    try:
        files.download(str(zip_dest))
    except Exception as e:
        print(f"Download trigger note: {e}. You can also right-click results.zip in the left Files panel to download.")